# Berlin neighborhood clustering – data exploration

This notebook builds and explores a neighborhood-level feature table for Berlin.

It combines:
- amenity counts from SQL
- accessibility signals
- spatial normalization using neighborhood area
- population normalization using population data
- proximity features based on centroid-to-amenity distance

The goal is to prepare a clean feature space for later clustering and neighborhood profiling.

In [1]:
from pathlib import Path
import os
import warnings
from dotenv import load_dotenv

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 120)

## Connect to the database

We connect to the local PostgreSQL instance.

The SQL logic is kept in separate `.sql` files so that:
- the notebook stays readable
- SQL can be tested independently
- feature engineering logic is easier to maintain

In [2]:
# Load database credentials from environment variables

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5433")
DB_NAME = os.getenv("DB_NAME")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Quick connection test
query_test = "SELECT CURRENT_DATABASE();"

try:
    with engine.connect() as conn:
        result = conn.execute(text(query_test))
        for row in result:
            print("Connected to database:", row[0])
except SQLAlchemyError as e:
    print("Database connection failed.")
    print(e)

Connected to database: layereddb


## Load SQL script to count amenities

This SQL script builds the base neighborhood amenity table.

It includes:
- amenity counts per neighborhood
- selected numeric aggregations such as bike lane length, park area, playground area, and average long-term rent
- selected sub-counts used later for accessibility scoring

This is the base table from which we derive:
- accessibility ratio
- spatial density features
- population density features

In [3]:
sql_counts_path = Path("../sql/create_neighborhood_amenity_counts.sql")

with open(sql_counts_path, "r", encoding="utf-8") as f:
    counts_query = f.read()

print(f"Loaded SQL from: {sql_counts_path}")

Loaded SQL from: ../sql/create_neighborhood_amenity_counts.sql


## Build amenity table from SQL counts

In [4]:
try:
    df_features = pd.read_sql(text(counts_query), engine)
    print("Feature table shape:", df_features.shape)
    display(df_features.head())
except SQLAlchemyError as e:
    print("Query execution failed.")
    print(e)

Feature table shape: (96, 82)


,neighborhood_id,neighborhood,district,atm_count,atm_accessible_count,bakery_count,bank_count,bank_accessible_count,bike_lane_length_m,bouldering_spot_count,bouldering_accessible_count,bus_stop_count,bus_stop_accessible_count,dental_office_count,dental_office_accessible_count,diplomatic_mission_count,doctor_count,emergency_station_count,police_station_count,fire_station_count,exhibition_center_count,food_market_count,food_market_accessible_count,gallery_count,gallery_accessible_count,gas_station_count,gas_station_electric_count,government_office_count,hospital_count,hospital_accessible_count,hotel_count,hotel_accessible_count,kindergarten_count,kindergarten_capacity,library_count,library_accessible_count,long_term_listing_count,long_term_avg_price_euro,mall_count,mall_accessible_count,milieuschutz_area_ha,museum_count,museum_accessible_count,night_club_count,night_club_accessible_count,parking_space_count,parking_capacity,parking_disabled_capacity,park_area_sq_m,petstore_count,pharmacy_count,pharmacy_accessible_count,playground_area_sq_m,pool_count,public_artwork_count,recycling_point_count,religious_institution_count,religious_institution_accessible_count,research_institute_count,research_institute_accessible_count,school_count,short_term_listing_count,social_club_count,spaeti_count,supermarket_count,theater_count,theater_accessible_count,cinema_count,stage_theater_count,tram_stop_count,tram_stop_accessible_count,transport_station_entrance_count,transport_station_entrance_accessible_count,transport_station_count,transport_station_accessible_count,university_count,university_accessible_count,venue_count,venue_accessible_count,veterinary_clinic_count,vocational_school_count,vocational_school_accessible_count
0,0401,Charlottenburg,Charlottenburg-Wilmersdorf,54,18,70,21,13,147145.855278,0,0,143,113,59,4,12,88,7,3,3,0,12,6,41,2,82,75,21,10,3,102,45,77,647,12,5,81,1501.555556,4,1,0.0,21,7,12,1,7627,33907.0,27.0,789441.514957,5,39,24,61142.768476,3,93,45,30,10,8,1,24,0,64,88,63,25,8,9,0,18,12,80,16,17,12,9,3,782,162,9,6,0
1,0406,Charlottenburg-Nord,Charlottenburg-Wilmersdorf,4,0,2,0,0,40413.196484,0,0,55,6,1,0,0,2,4,0,4,0,0,0,0,0,4,2,3,0,0,0,0,9,0,1,1,8,449.875000,0,0,0.0,1,0,1,0,1600,6642.0,7.0,82656.392816,0,3,2,18471.688716,1,5,13,3,1,0,0,6,0,8,1,4,0,0,0,0,3,1,9,2,3,1,0,0,9,2,0,0,0
2,0404,Grunewald,Charlottenburg-Wilmersdorf,0,0,1,2,1,66980.626977,1,0,43,1,2,0,25,5,0,0,0,0,0,0,0,0,5,3,3,1,0,7,2,6,0,2,0,10,2821.000000,0,0,0.0,1,0,0,0,3314,8844.0,4.0,166516.611549,0,1,1,6363.970608,6,7,6,3,0,1,0,4,71,8,1,1,1,0,0,0,1,1,0,0,1,1,0,0,18,2,1,1,0
3,0407,Halensee,Charlottenburg-Wilmersdorf,10,1,9,2,0,12257.866992,0,0,18,7,12,0,2,31,0,0,0,0,1,0,0,0,6,4,1,2,0,3,0,11,30,0,0,8,1251.250000,0,0,634.4,0,0,1,0,711,3906.0,0.0,20722.408622,1,4,4,7203.339367,0,6,7,4,0,0,0,1,74,4,7,8,1,0,0,0,1,1,1,0,2,2,0,0,44,6,2,0,0
4,0403,Schmargendorf,Charlottenburg-Wilmersdorf,2,0,6,4,2,50061.686708,0,0,32,3,3,0,8,5,0,0,0,0,1,0,0,0,9,9,2,2,1,2,0,10,310,1,1,17,1356.352941,0,0,0.0,0,0,1,0,2105,9779.0,4.0,45441.374835,1,7,2,13535.220541,2,3,14,5,2,0,0,7,39,5,7,9,0,0,0,0,0,0,0,0,0,0,0,0,33,8,2,0,0


## Basic checks

We inspect:
- shape
- data types
- missing values
- summary statistics

In [5]:
print("Rows:", df_features.shape[0])
print("Columns:", df_features.shape[1])

display(pd.DataFrame({
    "column": df_features.columns,
    "dtype": df_features.dtypes.astype(str).values
}))

missing = (
    df_features.isna()
    .sum()
    .reset_index()
    .rename(columns={"index": "column", 0: "missing_count"})
)

missing["missing_pct"] = (missing["missing_count"] / len(df_features) * 100).round(2)
missing = missing.sort_values(["missing_count", "column"], ascending=[False, True])

display(missing)

display(df_features.describe(include="all").T)

Rows: 96
Columns: 82


,column,dtype
0,neighborhood_id,str
1,neighborhood,str
2,district,str
3,atm_count,int64
4,atm_accessible_count,int64
5,bakery_count,int64
6,bank_count,int64
7,bank_accessible_count,int64
8,bike_lane_length_m,float64
9,bouldering_spot_count,int64


,column,missing_count,missing_pct
37,long_term_avg_price_euro,5,5.21
4,atm_accessible_count,0,0.00
3,atm_count,0,0.00
5,bakery_count,0,0.00
7,bank_accessible_count,0,0.00
6,bank_count,0,0.00
8,bike_lane_length_m,0,0.00
10,bouldering_accessible_count,0,0.00
9,bouldering_spot_count,0,0.00
12,bus_stop_accessible_count,0,0.00


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
neighborhood_id,96,96,0401,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
neighborhood,96,96,Charlottenburg,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
district,96,12,Treptow-Köpenick,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN
atm_count,96.0,NaN,NaN,NaN,11.0625,24.772571,0.0,1.0,3.0,7.25,117.0
atm_accessible_count,96.0,NaN,NaN,NaN,3.020833,6.402268,0.0,0.0,1.0,3.0,37.0
bakery_count,96.0,NaN,NaN,NaN,13.645833,19.745741,0.0,2.0,6.0,15.0,87.0
bank_count,96.0,NaN,NaN,NaN,2.489583,3.968282,0.0,0.0,1.0,3.25,23.0
bank_accessible_count,96.0,NaN,NaN,NaN,1.666667,2.827186,0.0,0.0,1.0,2.0,18.0
bike_lane_length_m,96.0,NaN,NaN,NaN,67242.205795,43652.234468,5092.801969,33369.27393,59937.111982,83574.569642,210128.03059
bouldering_spot_count,96.0,NaN,NaN,NaN,0.375,0.909019,0.0,0.0,0.0,0.0,5.0


In [6]:
output_counts_path = Path("../data/neighborhood_features_counts.csv")
output_counts_path.parent.mkdir(parents=True, exist_ok=True)

df_features.to_csv(output_counts_path, index=False)
print(f"Saved counts table to: {output_counts_path}")

Saved counts table to: ../data/neighborhood_features_counts.csv


## Create accessibility score

Accessibility is summarized into one neighborhood-level ratio.

It is computed as:

accessible facilities / total facilities

This gives one interpretable signal of how accessibility-friendly a neighborhood is overall.

All individual `_accessible_count` columns are consolidated into a single
`accessibility_ratio` feature.

After calculation, the original accessibility columns and helper fields are removed
to keep the modeling table clean.

In [7]:
df_model = df_features.copy()

# -----------------------------
# Build accessibility ratio
# -----------------------------

# Find all accessibility columns automatically
accessible_cols = [
    col for col in df_model.columns
    if col.endswith("_accessible_count")
]

# Match each accessible column to its total count column
pairs = []

for acc_col in accessible_cols:
    total_col = acc_col.replace("_accessible_count", "_count")
    if total_col in df_model.columns:
        pairs.append((total_col, acc_col))

print(f"Found {len(pairs)} accessibility pairs")

# Sum accessible and total facilities
accessible_sum = pd.Series(0, index=df_model.index, dtype=float)
total_sum = pd.Series(0, index=df_model.index, dtype=float)

for total_col, acc_col in pairs:
    accessible_sum += df_model[acc_col].fillna(0)
    total_sum += df_model[total_col].fillna(0)

# Add parking accessibility
if {
    "parking_disabled_capacity",
    "parking_capacity"
}.issubset(df_model.columns):

    accessible_sum += df_model["parking_disabled_capacity"].fillna(0)
    total_sum += df_model["parking_capacity"].fillna(0)

# Final ratio
df_model["accessibility_ratio"] = (
    accessible_sum / total_sum.replace(0, np.nan)
)

# -----------------------------
# Drop no-longer-needed columns
# -----------------------------

drop_cols = accessible_cols.copy()

if "parking_disabled_capacity" in df_model.columns:
    drop_cols.append("parking_disabled_capacity")

df_model = df_model.drop(columns=drop_cols)

print("Accessibility columns removed")
print("New shape:", df_model.shape)

display(
    df_model[["neighborhood", "accessibility_ratio"]].head()
)

Found 22 accessibility pairs
Accessibility columns removed
New shape: (96, 59)


,neighborhood,accessibility_ratio
0,Charlottenburg,0.013922
1,Charlottenburg-Nord,0.003415
2,Grunewald,0.001456
3,Halensee,0.005234
4,Schmargendorf,0.002330


## Load neighborhood area from geometry

Neighborhood area is needed to derive:
- spatial density features such as counts per km²
- land-use shares such as park share and playground share

The neighborhood geometry is stored as hex-encoded WKB, so it must be decoded before computing area.

In [8]:
query_area = """
SELECT
    LPAD(neighborhood_id::text, 4, '0') AS neighborhood_id,
    ST_Area(
        ST_Transform(
            ST_SetSRID(
                ST_GeomFromWKB(decode(geometry, 'hex')),
                4326
            ),
            25833
        )
    ) AS area_sq_m
FROM berlin_source_data.neighborhoods
WHERE geometry IS NOT NULL;
"""

df_area = pd.read_sql(text(query_area), engine)

display(df_area.head())
print("Area table shape:", df_area.shape)

,neighborhood_id,area_sq_m
0,0503,5.661235e+06
1,0504,1.088277e+07
2,0903,4.821521e+06
3,0904,6.518400e+06
4,0905,3.498748e+06


Area table shape: (96, 2)


In [9]:
# Merge area into the working feature table
df_model = df_model.merge(df_area, on="neighborhood_id", how="left")

print("Missing area values:", df_model["area_sq_m"].isna().sum())

# Convert to km² for density calculations
df_model["area_sq_km"] = df_model["area_sq_m"] / 1_000_000

display(
    df_model[["neighborhood_id", "neighborhood", "area_sq_m", "area_sq_km"]].head()
)

Missing area values: 0


,neighborhood_id,neighborhood,area_sq_m,area_sq_km
0,0401,Charlottenburg,1.059155e+07,10.591550
1,0406,Charlottenburg-Nord,6.196480e+06,6.196480
2,0404,Grunewald,2.234392e+07,22.343924
3,0407,Halensee,1.266255e+06,1.266255
4,0403,Schmargendorf,3.586366e+06,3.586366


## Calculate spatial density from counts

Some features are better interpreted relative to neighborhood area rather than population.

We use:
- counts per km² for point-based infrastructure and activities
- meters per km² for bike lanes

Area-based features are better expressed as a share of neighborhood area.

This applies to:
- park area
- playground area
- milieuschutz area

In [10]:
area_km_safe = df_model["area_sq_km"].replace(0, np.nan)
area_safe = df_model["area_sq_m"].replace(0, np.nan)

# -----------------------------
# Spatial density: counts per km²
# -----------------------------
spatial_density_features = [
    "bouldering_spot_count",
    "bus_stop_count",
    "cinema_count",
    "dental_office_count",
    "diplomatic_mission_count",
    "doctor_count",
    "emergency_station_count",
    "exhibition_center_count",
    "fire_station_count",
    "food_market_count",
    "gallery_count",
    "gas_station_count",
    "gas_station_electric_count",
    "hospital_count",
    "hotel_count",
    "kindergarten_count",
    "library_count",
    "long_term_listing_count",
    "mall_count",
    "museum_count",
    "night_club_count",
    "parking_capacity",
    "parking_space_count",
    "pharmacy_count",
    "police_station_count",
    "pool_count",
    "public_artwork_count",
    "religious_institution_count",
    "research_institute_count",
    "school_count",
    "short_term_listing_count",
    "social_club_count",
    "spaeti_count",
    "stage_theater_count",
    "supermarket_count",
    "theater_count",
    "tram_stop_count",
    "transport_station_count",
    "transport_station_entrance_count",
    "university_count",
    "venue_count",
    "vocational_school_count",
]

for col in spatial_density_features:
    if col in df_model.columns:
        df_model[f"{col}_per_sq_km"] = df_model[col] / area_km_safe

# -----------------------------
# Linear infrastructure per km²
# -----------------------------
if "bike_lane_length_m" in df_model.columns:
    df_model["bike_lane_length_m_per_sq_km"] = df_model["bike_lane_length_m"] / area_km_safe

# -----------------------------
# Area-share features
# -----------------------------
if "park_area_sq_m" in df_model.columns:
    df_model["park_share"] = df_model["park_area_sq_m"] / area_safe

if "playground_area_sq_m" in df_model.columns:
    df_model["playground_share"] = df_model["playground_area_sq_m"] / area_safe

if "milieuschutz_area_ha" in df_model.columns:
    df_model["milieuschutz_area_sq_m"] = df_model["milieuschutz_area_ha"] * 10_000
    df_model["milieuschutz_share"] = df_model["milieuschutz_area_sq_m"] / area_safe

In [11]:
display(
    df_model[[
        "neighborhood",
        "area_sq_km",
        "bike_lane_length_m_per_sq_km",
        "park_share",
        "playground_share"
    ]]
    .sort_values("bike_lane_length_m_per_sq_km", ascending=False)
    .head(10)
)

,neighborhood,area_sq_km,bike_lane_length_m_per_sq_km,park_share,playground_share
75,Friedenau,1.654282,20108.370165,0.021242,0.007025
6,Wilmersdorf,7.157517,18126.833482,0.029784,0.011321
25,Hansaviertel,0.528038,15545.929000,0.028952,0.004254
58,Wittenau,5.893308,14821.009063,0.067348,0.001973
72,Steglitz,6.782407,14508.992429,0.030859,0.007312
20,Hellersdorf,8.142248,14432.593111,0.135698,0.012833
43,Prenzlauer Berg,11.003473,14126.397098,0.086650,0.010719
4,Schmargendorf,3.586366,13958.890970,0.012671,0.003774
0,Charlottenburg,10.591550,13892.759605,0.074535,0.005773
26,Mitte,10.673582,13006.438505,0.061526,0.007617


## Import population data and calculate population density features

For resident-facing services, counts are more meaningful relative to population.

We therefore calculate counts per 1000 residents for selected features.

In [12]:
pop_path = Path("../data/population_data.csv")
df_pop = pd.read_csv(pop_path, dtype={"neighborhood_id": str})

# Normalize neighborhood_id
df_pop["neighborhood_id"] = df_pop["neighborhood_id"].str.zfill(4)

# Keep only total population
df_pop = df_pop.rename(columns={"Insgesamt": "population"})
df_pop = df_pop[["neighborhood_id", "population"]].copy()

display(df_pop.head())

,neighborhood_id,population
0,0101,108953
1,0102,84913
2,0103,6120
3,0104,16217
4,0105,86662


In [13]:
df_model = df_model.merge(df_pop, on="neighborhood_id", how="left")

print("Missing population values:", df_model["population"].isna().sum())

pop_safe = df_model["population"].replace(0, np.nan)

population_density_features = [
    "kindergarten_count",
    "school_count",
    "vocational_school_count",
    "kindergarten_capacity",
    "long_term_listing_count",
]

for col in population_density_features:
    if col in df_model.columns:
        df_model[f"{col}_per_1000"] = df_model[col] / pop_safe * 1000

Missing population values: 0


## Load SQL script for proximity features

This SQL script computes centroid-based proximity features.

For each neighborhood, it calculates distance to nearby amenities.
Different values of k are used depending on amenity frequency:

- 5 nearest for very common amenities
- 3 nearest for medium-frequency amenities
- 1 nearest for rare / critical amenities

In [14]:
sql_distance_path = Path("../sql/create_neighborhood_proximity_features.sql")

with open(sql_distance_path, "r", encoding="utf-8") as f:
    distance_query = f.read()

print(f"Loaded SQL from: {sql_distance_path}")

Loaded SQL from: ../sql/create_neighborhood_proximity_features.sql


In [15]:
try:
    df_distance = pd.read_sql(text(distance_query), engine)
    print("Distance table shape:", df_distance.shape)
    display(df_distance.head())
except SQLAlchemyError as e:
    print("Distance query execution failed.")
    print(e)

Distance table shape: (96, 34)


,neighborhood_id,neighborhood,district,avg_dist_5_atm_m,avg_dist_5_bakery_m,avg_dist_5_bus_stop_m,avg_dist_3_dental_office_m,avg_dist_3_doctor_m,dist_1_hospital_m,avg_dist_3_kindergarten_m,avg_dist_3_park_m,avg_dist_5_pharmacy_m,avg_dist_3_playground_m,avg_dist_3_school_m,avg_dist_5_spaeti_m,avg_dist_5_supermarket_m,avg_dist_5_tram_stop_m,avg_dist_5_transport_station_m,avg_dist_3_bank_m,dist_1_emergency_station_m,dist_1_fire_station_m,avg_dist_3_food_market_m,avg_dist_3_gas_station_m,avg_dist_3_gas_station_electric_m,avg_dist_3_government_office_m,avg_dist_3_library_m,avg_dist_3_mall_m,avg_dist_3_petstore_m,dist_1_police_station_m,avg_dist_3_pool_m,avg_dist_3_recycling_point_m,avg_dist_3_transport_station_entrance_m,avg_dist_3_veterinary_clinic_m,avg_dist_3_vocational_school_m
0,0101,Mitte,Mitte,326.038650,356.257021,425.035761,772.790702,472.433775,524.281265,392.539379,75.998957,600.227801,244.327377,515.916098,297.292779,373.075277,293.803482,547.573391,559.463390,819.911240,828.388675,335.701826,381.193452,381.193452,520.474876,275.745301,996.883549,1865.975961,819.911240,763.722499,312.912632,422.334341,1002.916412,822.061016
1,0102,Moabit,Mitte,311.582538,360.599120,255.837492,240.894515,404.832017,481.050192,137.007510,399.605036,393.803474,328.012033,549.052821,218.934356,247.095059,384.167323,744.922704,486.500075,852.964674,1008.801233,339.497283,235.942139,235.942139,209.488920,899.875920,927.429918,2516.887471,852.964674,1324.623517,450.935891,347.844116,475.731932,707.314346
2,0103,Hansaviertel,Mitte,659.508177,547.444931,180.581522,783.459897,262.196795,742.482102,306.874703,32.998188,640.210126,171.504319,353.134447,464.114161,503.059441,658.357306,686.909446,855.547337,736.013558,736.013558,574.292738,155.765166,221.778465,588.145734,736.390035,1347.176140,2171.435808,1256.338330,1603.230536,116.692844,591.572178,647.720211,824.276005
3,0104,Tiergarten,Mitte,836.210663,933.855900,354.325274,1230.178684,1081.944780,810.322465,711.451802,307.297730,1186.529734,601.451386,630.848959,941.414009,1060.857267,1285.454555,1275.177209,1287.664733,846.709515,1360.120819,1104.725844,535.283860,535.283860,518.456984,780.420533,1411.977642,1803.987940,1389.744218,1481.619035,862.701946,1107.791604,1230.649688,1303.941450
4,0105,Wedding,Mitte,433.533930,399.588735,315.864620,499.645826,458.259550,822.231913,133.524096,418.381664,629.326911,120.879971,665.181242,313.722655,461.582683,605.191448,998.995395,752.326393,753.817839,753.817839,874.481971,309.676630,309.676630,800.208995,1101.069208,1528.828100,1506.288176,1310.785800,1514.665417,269.387257,588.989041,805.480545,975.045973


## Merge counts and proximity features

In [16]:
df_model = df_model.merge(
    df_distance,
    on=["neighborhood_id", "neighborhood", "district"],
    how="left"
)

print("Final merged table shape:", df_model.shape)
display(df_model.head())

Final merged table shape: (96, 145)


,neighborhood_id,neighborhood,district,atm_count,bakery_count,bank_count,bike_lane_length_m,bouldering_spot_count,bus_stop_count,dental_office_count,diplomatic_mission_count,doctor_count,emergency_station_count,police_station_count,fire_station_count,exhibition_center_count,food_market_count,gallery_count,gas_station_count,gas_station_electric_count,government_office_count,hospital_count,hotel_count,kindergarten_count,kindergarten_capacity,library_count,long_term_listing_count,long_term_avg_price_euro,mall_count,milieuschutz_area_ha,museum_count,night_club_count,parking_space_count,parking_capacity,park_area_sq_m,petstore_count,pharmacy_count,playground_area_sq_m,pool_count,public_artwork_count,recycling_point_count,religious_institution_count,research_institute_count,school_count,short_term_listing_count,social_club_count,spaeti_count,supermarket_count,theater_count,cinema_count,stage_theater_count,tram_stop_count,transport_station_entrance_count,transport_station_count,university_count,venue_count,veterinary_clinic_count,vocational_school_count,accessibility_ratio,area_sq_m,area_sq_km,bouldering_spot_count_per_sq_km,bus_stop_count_per_sq_km,cinema_count_per_sq_km,dental_office_count_per_sq_km,diplomatic_mission_count_per_sq_km,doctor_count_per_sq_km,emergency_station_count_per_sq_km,exhibition_center_count_per_sq_km,fire_station_count_per_sq_km,food_market_count_per_sq_km,gallery_count_per_sq_km,gas_station_count_per_sq_km,gas_station_electric_count_per_sq_km,hospital_count_per_sq_km,hotel_count_per_sq_km,kindergarten_count_per_sq_km,library_count_per_sq_km,long_term_listing_count_per_sq_km,mall_count_per_sq_km,museum_count_per_sq_km,night_club_count_per_sq_km,parking_capacity_per_sq_km,parking_space_count_per_sq_km,pharmacy_count_per_sq_km,police_station_count_per_sq_km,pool_count_per_sq_km,public_artwork_count_per_sq_km,religious_institution_count_per_sq_km,research_institute_count_per_sq_km,school_count_per_sq_km,short_term_listing_count_per_sq_km,social_club_count_per_sq_km,spaeti_count_per_sq_km,stage_theater_count_per_sq_km,supermarket_count_per_sq_km,theater_count_per_sq_km,tram_stop_count_per_sq_km,transport_station_count_per_sq_km,transport_station_entrance_count_per_sq_km,university_count_per_sq_km,venue_count_per_sq_km,vocational_school_count_per_sq_km,bike_lane_length_m_per_sq_km,park_share,playground_share,milieuschutz_area_sq_m,milieuschutz_share,population,kindergarten_count_per_1000,school_count_per_1000,vocational_school_count_per_1000,kindergarten_capacity_per_1000,long_term_listing_count_per_1000,avg_dist_5_atm_m,avg_dist_5_bakery_m,avg_dist_5_bus_stop_m,avg_dist_3_dental_office_m,avg_dist_3_doctor_m,dist_1_hospital_m,avg_dist_3_kindergarten_m,avg_dist_3_park_m,avg_dist_5_pharmacy_m,avg_dist_3_playground_m,avg_dist_3_school_m,avg_dist_5_spaeti_m,avg_dist_5_supermarket_m,avg_dist_5_tram_stop_m,avg_dist_5_transport_station_m,avg_dist_3_bank_m,dist_1_emergency_station_m,dist_1_fire_station_m,avg_dist_3_food_market_m,avg_dist_3_gas_station_m,avg_dist_3_gas_station_electric_m,avg_dist_3_government_office_m,avg_dist_3_library_m,avg_dist_3_mall_m,avg_dist_3_petstore_m,dist_1_police_station_m,avg_dist_3_pool_m,avg_dist_3_recycling_point_m,avg_dist_3_transport_station_entrance_m,avg_dist_3_veterinary_clinic_m,avg_dist_3_vocational_school_m
0,0401,Charlottenburg,Charlottenburg-Wilmersdorf,54,70,21,147145.855278,0,143,59,12,88,7,3,3,0,12,41,82,75,21,10,102,77,647,12,81,1501.555556,4,0.0,21,12,7627,33907.0,789441.514957,5,39,61142.768476,3,93,45,30,8,24,0,64,88,63,25,9,0,18,80,17,9,782,9,6,0.013922,1.059155e+07,10.591550,0.000000,13.501329,0.849734,5.570478,1.132979,8.308510,0.660904,0.0,0.283245,1.132979,3.87101,7.742021,7.081117,0.944149,9.630319,7.269946,1.132979,7.647606,0.37766,1.982713,1.132979,3201.325644,720.102359,3.682181,0.283245,0.283245,8.780585,2.832447,0.755319,2.265957,0.000000,6.042553,8.308510,0.0,5.948138,2.360372,1.699468,1.605053,7.553191,0.849734,73.832443,0.566489,13892.759605,0.074535,0.005773,

## Final feature selection and model table cleanup

At this stage, the dataset contains both:

- raw source variables (for example raw counts and helper columns)
- engineered modeling features (density, shares, proximity)

Before export, we simplify the table by removing raw variables that are now fully represented by engineered features.

### What gets removed
We drop:
- all raw `*_count` columns
- absolute capacity helper columns already normalized
- area helper columns
- population helper column

### What stays
We keep:
- identifiers
- key raw scalar features
- spatial density features
- population-normalized features
- area-share features
- proximity features

This creates the final ML-ready feature table.

In [17]:
# Inspect all current columns with index
for i, col in enumerate(df_model.columns):
    print(f"{i}: {col}")

0: neighborhood_id
1: neighborhood
2: district
3: atm_count
4: bakery_count
5: bank_count
6: bike_lane_length_m
7: bouldering_spot_count
8: bus_stop_count
9: dental_office_count
10: diplomatic_mission_count
11: doctor_count
12: emergency_station_count
13: police_station_count
14: fire_station_count
15: exhibition_center_count
16: food_market_count
17: gallery_count
18: gas_station_count
19: gas_station_electric_count
20: government_office_count
21: hospital_count
22: hotel_count
23: kindergarten_count
24: kindergarten_capacity
25: library_count
26: long_term_listing_count
27: long_term_avg_price_euro
28: mall_count
29: milieuschutz_area_ha
30: museum_count
31: night_club_count
32: parking_space_count
33: parking_capacity
34: park_area_sq_m
35: petstore_count
36: pharmacy_count
37: playground_area_sq_m
38: pool_count
39: public_artwork_count
40: recycling_point_count
41: religious_institution_count
42: research_institute_count
43: school_count
44: short_term_listing_count
45: social_clu

In [18]:
# -----------------------------------
# Inspect columns to be dropped
# -----------------------------------

# 1. All raw count columns
count_cols_to_drop = [
    col for col in df_model.columns
    if col.endswith("_count")
]

# 2. Additional raw helper / absolute columns
extra_cols_to_drop = [
    "bike_lane_length_m",
    "milieuschutz_area_ha",
    "park_area_sq_m",
    "playground_area_sq_m",
    "kindergarten_capacity",
    "parking_capacity",
    "area_sq_m",
    "area_sq_km",
    "milieuschutz_area_sq_m",
    "population",
]

# 3. Final drop list
cols_to_drop = count_cols_to_drop + [
    col for col in extra_cols_to_drop
    if col in df_model.columns
]

drop_preview = pd.DataFrame({
    "column_to_drop": cols_to_drop
})

print(f"Number of columns to drop: {len(cols_to_drop)}")
display(drop_preview)

Number of columns to drop: 58


,column_to_drop
0,atm_count
1,bakery_count
2,bank_count
3,bouldering_spot_count
4,bus_stop_count
5,dental_office_count
6,diplomatic_mission_count
7,doctor_count
8,emergency_station_count
9,police_station_count


In [19]:
# -----------------------------------
# Inspect columns that will stay
# -----------------------------------

remaining_cols = [
    col for col in df_model.columns
    if col not in cols_to_drop
]

keep_preview = pd.DataFrame({
    "column_to_keep": remaining_cols
})

print(f"Number of columns to keep: {len(remaining_cols)}")
display(keep_preview)

Number of columns to keep: 87


,column_to_keep
0,neighborhood_id
1,neighborhood
2,district
3,long_term_avg_price_euro
4,accessibility_ratio
5,bouldering_spot_count_per_sq_km
6,bus_stop_count_per_sq_km
7,cinema_count_per_sq_km
8,dental_office_count_per_sq_km
9,diplomatic_mission_count_per_sq_km


In [20]:
# -----------------------------------
# Drop raw columns from modeling table
# -----------------------------------

df_model = df_model.drop(
    columns=cols_to_drop,
    errors="ignore"
)

print("Columns dropped successfully")
print("Final shape:", df_model.shape)

display(pd.DataFrame({
    "remaining_column": df_model.columns
}))

Columns dropped successfully
Final shape: (96, 87)


,remaining_column
0,neighborhood_id
1,neighborhood
2,district
3,long_term_avg_price_euro
4,accessibility_ratio
5,bouldering_spot_count_per_sq_km
6,bus_stop_count_per_sq_km
7,cinema_count_per_sq_km
8,dental_office_count_per_sq_km
9,diplomatic_mission_count_per_sq_km


In [21]:
df_model.describe(include="all")

,neighborhood_id,neighborhood,district,long_term_avg_price_euro,accessibility_ratio,bouldering_spot_count_per_sq_km,bus_stop_count_per_sq_km,cinema_count_per_sq_km,dental_office_count_per_sq_km,diplomatic_mission_count_per_sq_km,doctor_count_per_sq_km,emergency_station_count_per_sq_km,exhibition_center_count_per_sq_km,fire_station_count_per_sq_km,food_market_count_per_sq_km,gallery_count_per_sq_km,gas_station_count_per_sq_km,gas_station_electric_count_per_sq_km,hospital_count_per_sq_km,hotel_count_per_sq_km,kindergarten_count_per_sq_km,library_count_per_sq_km,long_term_listing_count_per_sq_km,mall_count_per_sq_km,museum_count_per_sq_km,night_club_count_per_sq_km,parking_capacity_per_sq_km,parking_space_count_per_sq_km,pharmacy_count_per_sq_km,police_station_count_per_sq_km,pool_count_per_sq_km,public_artwork_count_per_sq_km,religious_institution_count_per_sq_km,research_institute_count_per_sq_km,school_count_per_sq_km,short_term_listing_count_per_sq_km,social_club_count_per_sq_km,spaeti_count_per_sq_km,stage_theater_count_per_sq_km,supermarket_count_per_sq_km,theater_count_per_sq_km,tram_stop_count_per_sq_km,transport_station_count_per_sq_km,transport_station_entrance_count_per_sq_km,university_count_per_sq_km,venue_count_per_sq_km,vocational_school_count_per_sq_km,bike_lane_length_m_per_sq_km,park_share,playground_share,milieuschutz_share,kindergarten_count_per_1000,school_count_per_1000,vocational_school_count_per_1000,kindergarten_capacity_per_1000,long_term_listing_count_per_1000,avg_dist_5_atm_m,avg_dist_5_bakery_m,avg_dist_5_bus_stop_m,avg_dist_3_dental_office_m,avg_dist_3_doctor_m,dist_1_hospital_m,avg_dist_3_kindergarten_m,avg_dist_3_park_m,avg_dist_5_pharmacy_m,avg_dist_3_playground_m,avg_dist_3_school_m,avg_dist_5_spaeti_m,avg_dist_5_supermarket_m,avg_dist_5_tram_stop_m,avg_dist_5_transport_station_m,avg_dist_3_bank_m,dist_1_emergency_station_m,dist_1_fire_station_m,avg_dist_3_food_market_m,avg_dist_3_gas_station_m,avg_dist_3_gas_station_electric_m,avg_dist_3_government_office_m,avg_dist_3_library_m,avg_dist_3_mall_m,avg_dist_3_petstore_m,dist_1_police_station_m,avg_dist_3_pool_m,avg_dist_3_recycling_point_m,avg_dist_3_transport_station_entrance_m,avg_dist_3_veterinary_clinic_m,avg_dist_3_vocational_school_m
count,96,96,96,91.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.0,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000,96.000000
unique,96,96,12,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,0401,Charlottenburg,Treptow-Köpenick,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,1,1,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN

## Correlation analysis by feature family

At this stage, the feature table contains three distinct groups of engineered variables:

- **spatial features**  
  amenities normalized by neighborhood area  
  (counts per km², shares, infrastructure density)

- **population-normalized features**  
  amenities normalized by resident population  
  (counts per 1,000 residents)

- **distance-based features**  
  accessibility measures from neighborhood centroid  
  (distance to nearest 1, 3, or 5 amenities)

A single correlation matrix across all features would be difficult to interpret because these groups represent different urban dimensions and operate on different scales.

For example:

- high amenity density often implies lower distance
- population-normalized metrics capture residential service intensity
- spatial density captures land-use concentration

Combining all of them into one matrix would make the strongest spatial signals dominate and may hide meaningful patterns in population or accessibility features.

For this reason, correlation analysis is performed **separately by feature family**.

### 1. Spatial correlation matrix

This matrix evaluates how amenities and infrastructure co-locate spatially across neighborhoods.

It helps identify urban clusters such as:

- healthcare corridors
- cultural districts
- nightlife hubs
- transport-rich areas

Examples of questions addressed:

- Do pharmacies and supermarkets cluster together?
- Are tram stops associated with cultural venues?
- Do parks and playgrounds co-occur?

In [22]:
spatial_cols = [
    col for col in df_model.columns
    if col.endswith("_per_sq_km")
    or col in [
        "park_share",
        "playground_share",
        "milieuschutz_share",
        "bike_lane_length_m_per_sq_km"
    ]
]

corr_spatial = df_model[spatial_cols].corr()

display(corr_spatial)

,bouldering_spot_count_per_sq_km,bus_stop_count_per_sq_km,cinema_count_per_sq_km,dental_office_count_per_sq_km,diplomatic_mission_count_per_sq_km,doctor_count_per_sq_km,emergency_station_count_per_sq_km,exhibition_center_count_per_sq_km,fire_station_count_per_sq_km,food_market_count_per_sq_km,gallery_count_per_sq_km,gas_station_count_per_sq_km,gas_station_electric_count_per_sq_km,hospital_count_per_sq_km,hotel_count_per_sq_km,kindergarten_count_per_sq_km,library_count_per_sq_km,long_term_listing_count_per_sq_km,mall_count_per_sq_km,museum_count_per_sq_km,night_club_count_per_sq_km,parking_capacity_per_sq_km,parking_space_count_per_sq_km,pharmacy_count_per_sq_km,police_station_count_per_sq_km,pool_count_per_sq_km,public_artwork_count_per_sq_km,religious_institution_count_per_sq_km,research_institute_count_per_sq_km,school_count_per_sq_km,short_term_listing_count_per_sq_km,social_club_count_per_sq_km,spaeti_count_per_sq_km,stage_theater_count_per_sq_km,supermarket_count_per_sq_km,theater_count_per_sq_km,tram_stop_count_per_sq_km,transport_station_count_per_sq_km,transport_station_entrance_count_per_sq_km,university_count_per_sq_km,venue_count_per_sq_km,vocational_school_count_per_sq_km,bike_lane_length_m_per_sq_km,park_share,playground_share,milieuschutz_share
bouldering_spot_count_per_sq_km,1.000000,0.291310,0.087169,0.090185,-0.042928,0.059271,0.188273,-0.037365,0.063634,0.139375,0.250038,0.293468,0.284494,0.098461,0.187921,0.333796,0.112776,0.360551,0.226687,0.146933,0.154448,0.230144,0.053820,0.299367,0.240784,0.106020,0.257624,0.373765,0.033115,0.366469,-0.110636,0.452878,0.475341,NaN,0.305633,0.151243,0.206387,0.096836,0.136790,0.066769,0.226587,0.175930,0.157494,0.008798,0.465935,0.067368
bus_stop_count_per_sq_km,0.291310,1.000000,0.447825,0.520623,0.269112,0.516737,0.381626,-0.032684,0.136762,0.495754,0.428328,0.576563,0.548851,0.350096,0.457610,0.675426,0.402229,0.492204,0.308693,0.376599,0.360650,0.710398,0.665393,0.704381,0.341371,0.164546,0.240911,0.703970,0.110242,0.645768,0.245100,0.578933,0.503242,NaN,0.687610,0.505791,0.111660,0.470022,0.560121,0.207548,0.542859,0.456570,0.683461,0.069950,0.619388,0.286462
cinema_count_per_sq_km,0.087169,0.447825,1.000000,0.663920,0.350300,0.623966,0.466355,-0.057497,0.073684,0.540470,0.718314,0.682140,0.728079,0.319475,0.652601,0.705362,0.422257,0.766676,0.351165,0.622207,0.607317,0.550961,0.399808,0.694264,0.422416,0.206436,0.346981,0.680446,0.250478,0.533636,0.241092,0.761745,0.720417,NaN,0.745780,0.870517,0.395174,0.423081,0.810005,0.227271,0.823378,0.658890,0.471473,0.143281,0.473929,-0.027512
dental_office_count_per_sq_km,0.090185,0.520623,0.663920,1.000000,0.206176,0.967675,0.293293,-0.034387,0.021394,0.428736,0.343217,0.517799,0.530080,0.281437,0.390784,0.837720,0.116350,0.743285,0.269540,0.287348,0.236109,0.631302,0.543567,0.741184,0.118541,0.092973,0.117495,0.682612,0.071537,0.473954,0.674395,0.596730,0.517656,NaN,0.774464,0.634874,0.184381,0.385105,0.674862,0.094386,0.621522,0.596747,0.562060,-0.005436,0.368479,0.147467
diplomatic_mission_count_per_sq_km,-0.042928,0.269112,0.350300,0.206176,1.000000,0.227673,0.074862,0.020116,-0.101875,0.317018,0.576524,0.320564,0.336845,0.206985,0.598803,0.171061,0.549693,0.257194,0.129789,0.628168,0.376894,0.172685,0.081056,0.247151,0.102548,0.007592,0.580629,0.293730,0.203779,0.173359,0.131207,0.219542,0.216102,NaN,0.214022,0.562566,0.186113,0.383525,0.461642,0.345706,0.442057,0.153817,0.289146,-0.005362,0.094095,-0.009781
doctor_count_per_sq_km,0.059271,0.516737,0.623966,0.967675,0.227673,1.000000,0.242923,-0.044690,-0.049576,0.477244,0.325474,0.552622,0.556082,0.313464,0.341621,0.828426,0.169124,0.723872,0.273850,0.255911,0.238301,0.625394,0.511962,0.744388,0.131952,0.119652,0.179368,0.702883,0.049861,0.468423,0.722300,0.577593,0.504259,NaN,0.765640,0.643882,0.204121,0.479043,0.617815,0.142163,0.596493,0.527313,0.546894,0.020798,0.388595,0.135269
emergency_station_count_per_sq_km,0.188273,0.381626,0.466355,0.293293,0.074862,0.2429

### 2. Population-normalized correlation matrix

This matrix focuses on service availability relative to the number of residents.

It is particularly useful for understanding:

- family-oriented neighborhoods
- educational service intensity
- housing pressure indicators

Examples of questions addressed:

- Do neighborhoods with more schools per resident also have more kindergartens?
- Does long-term housing supply scale with family services?

In [23]:
population_cols = [
    col for col in df_model.columns
    if col.endswith("_per_1000")
]

corr_population = df_model[population_cols].corr()

display(corr_population)

,kindergarten_count_per_1000,school_count_per_1000,vocational_school_count_per_1000,kindergarten_capacity_per_1000,long_term_listing_count_per_1000
kindergarten_count_per_1000,1.000000,0.568598,-0.043677,-0.020165,-0.054592
school_count_per_1000,0.568598,1.000000,-0.066872,-0.077241,-0.099807
vocational_school_count_per_1000,-0.043677,-0.066872,1.000000,-0.167645,-0.060089
kindergarten_capacity_per_1000,-0.020165,-0.077241,-0.167645,1.000000,-0.011747
long_term_listing_count_per_1000,-0.054592,-0.099807,-0.060089,-0.011747,1.000000


### 3. Distance-based correlation matrix

This matrix focuses on accessibility patterns.

It helps identify neighborhoods with similar proximity profiles.

Examples of questions addressed:

- Do neighborhoods close to supermarkets also tend to be close to pharmacies?
- Is transport accessibility linked to healthcare accessibility?
- Which services tend to be jointly accessible?

This separation improves interpretability and allows each urban dimension to be analyzed on its own terms.

In [24]:
distance_cols = [
    col for col in df_model.columns
    if "_dist_" in col or col.startswith("dist_")
]

corr_distance = df_model[distance_cols].corr()

display(corr_distance)

,avg_dist_5_atm_m,avg_dist_5_bakery_m,avg_dist_5_bus_stop_m,avg_dist_3_dental_office_m,avg_dist_3_doctor_m,dist_1_hospital_m,avg_dist_3_kindergarten_m,avg_dist_3_park_m,avg_dist_5_pharmacy_m,avg_dist_3_playground_m,avg_dist_3_school_m,avg_dist_5_spaeti_m,avg_dist_5_supermarket_m,avg_dist_5_tram_stop_m,avg_dist_5_transport_station_m,avg_dist_3_bank_m,dist_1_emergency_station_m,dist_1_fire_station_m,avg_dist_3_food_market_m,avg_dist_3_gas_station_m,avg_dist_3_gas_station_electric_m,avg_dist_3_government_office_m,avg_dist_3_library_m,avg_dist_3_mall_m,avg_dist_3_petstore_m,dist_1_police_station_m,avg_dist_3_pool_m,avg_dist_3_recycling_point_m,avg_dist_3_transport_station_entrance_m,avg_dist_3_veterinary_clinic_m,avg_dist_3_vocational_school_m
avg_dist_5_atm_m,1.000000,0.862774,0.454824,0.768263,0.666633,0.594420,0.574422,0.634433,0.825867,0.766603,0.645310,0.834225,0.740314,0.684801,0.862313,0.646765,0.304326,0.192422,0.796090,0.700526,0.732726,0.766738,0.765160,0.730469,0.808409,0.606905,0.298956,0.536953,0.746815,0.706294,0.581015
avg_dist_5_bakery_m,0.862774,1.000000,0.510019,0.887182,0.776717,0.694880,0.683941,0.736683,0.919639,0.874769,0.783727,0.843779,0.869094,0.603644,0.859254,0.796239,0.358539,0.211349,0.792153,0.747816,0.799532,0.838531,0.829817,0.770401,0.808801,0.736655,0.304186,0.629156,0.688806,0.792981,0.646980
avg_dist_5_bus_stop_m,0.454824,0.510019,1.000000,0.485330,0.455233,0.319616,0.738320,0.556018,0.512973,0.578820,0.524870,0.413324,0.625554,0.316882,0.396614,0.439126,0.408295,0.359266,0.424086,0.559791,0.455473,0.426251,0.394387,0.448184,0.436780,0.317630,-0.014471,0.599051,0.421309,0.494536,0.307652
avg_dist_3_dental_office_m,0.768263,0.887182,0.485330,1.000000,0.821522,0.733934,0.674089,0.713973,0.900262,0.787762,0.794443,0.772140,0.847491,0.457576,0.802998,0.864020,0.286271,0.199725,0.792601,0.723788,0.807181,0.783672,0.831421,0.742400,0.724510,0.703816,0.257556,0.529450,0.656634,0.815028,0.663519
avg_dist_3_doctor_m,0.666633,0.776717,0.455233,0.821522,1.000000,0.737669,0.571400,0.652604,0.806949,0.746353,0.706342,0.587193,0.834911,0.361018,0.710768,0.791115,0.297084,0.168791,0.680016,0.651773,0.716162,0.666897,0.683535,0.590337,0.510498,0.654849,0.153307,0.540839,0.597413,0.704728,0.555848
dist_1_hospital_m,0.594420,0.694880,0.319616,0.733934,0.737669,1.000000,0.495746,0.563583,0.779610,0.644920,0.627103,0.566931,0.688563,0.341016,0.640057,0.750828,0.208393,0.145693,0.627913,0.497980,0.639835,0.671081,0.699303,0.576057,0.510423,0.568985,0.193378,0.484220,0.574130,0.681489,0.521624
avg_dist_3_kindergarten_m,0.574422,0.683941,0.738320,0.674089,0.571400,0.495746,1.000000,0.671050,0.668614,0.768295,0.747107,0.537549,0.749981,0.377741,0.556134,0.617022,0.355648,0.290004,0.553409,0.581791,0.549235,0.554704,0.578865,0.531833,0.560630,0.449114,0.188239,0.588631,0.542150,0.583040,0.435338
avg_dist_3_park_m,0.634433,0.736683,0.556018,0.713973,0.652604,0.563583,0.671050,1.000000,0.723534,0.775506,0.742006,0.649727,0.763834,0.443549,0.696923,0.760164,0.315449,0.190232,0.677850,0.677201,0.656527,0.721802,0.756640,0.678219,0.642460,0.687048,0.276964,0.577552,0.515255,0.700321,0.620826
avg_dist_5_pharmacy_m,0.825867,0.919639,0.512973,0.900262,0.806949,0.779610,0.668614,0.723534,1.000000,0.840647,0.780737,0.813551,0.903750,0.517438,0.870161,0.864074,0.275443,0.154254,0.826753,0.664504,0.740815,0.825378,0.842795,0.775442,0.769971,0.738331,0.257621,0.620390,0.715967,0.842567,0.673619
avg_dist_3_playground_m,0.766603,0.874769,0.578820,0.787762,0.746353,0.644920,0.768295,0.775506,0.840647,1.000000,0.804592,0.688609,0.878610,0.449614,0.751035,0.748249,0.374718,0.246948,0.677859,0.699667,0.726974,0.734237,0.769622,0.678272,0.744872,0.644523,0.189896,0.626104,0.580488,0.729848,0.585865


## Extracting strongest correlation pairs

While full correlation matrices are useful for broad exploration, they can become difficult to read when working with many engineered features.

To improve interpretability, the next step converts each correlation matrix into a **long-format table of the strongest correlation pairs**.

This process:

- reshapes the correlation matrix into pairwise rows
- removes self-correlations  
  (a feature correlated with itself)
- removes duplicated mirrored pairs  
  (for example A–B and B–A)
- keeps only correlations above a chosen threshold
- sorts results from strongest to weakest

This allows us to focus on the most meaningful relationships within each feature family.

---

### Correlation thresholds

Different thresholds are used depending on the feature group:

- **spatial features:** 0.7  
  strong co-location patterns

- **population-normalized features:** 0.4  
  fewer variables and typically weaker expected relationships

- **distance features:** 0.7  
  strong accessibility clusters

These tables provide a much clearer basis for interpretation than scanning the full matrices.

They help identify:

- functional urban clusters
- co-accessibility patterns
- residential service relationships
- potential redundant features for modeling

In [25]:
def get_top_correlations(corr_matrix, min_corr=0.7):
    corr_long = (
        corr_matrix
        .stack()
        .reset_index()
    )

    corr_long.columns = ["feature_1", "feature_2", "correlation"]

    corr_long = corr_long[
        corr_long["feature_1"] != corr_long["feature_2"]
    ]

    corr_long["pair_key"] = corr_long.apply(
        lambda x: tuple(sorted([x["feature_1"], x["feature_2"]])),
        axis=1
    )

    corr_long = corr_long.drop_duplicates("pair_key")

    corr_long = corr_long[
        corr_long["correlation"].abs() >= min_corr
    ].sort_values("correlation", ascending=False)

    return corr_long.drop(columns="pair_key")

In [26]:
top_spatial = get_top_correlations(corr_spatial)
top_population = get_top_correlations(corr_population, min_corr=0.4)
top_distance = get_top_correlations(corr_distance)

display(top_spatial.head(30))
display(top_population.head(30))
display(top_distance.head(30))

,feature_1,feature_2,correlation
518,gas_station_count_per_sq_km,gas_station_electric_count_per_sq_km,0.988443
143,dental_office_count_per_sq_km,doctor_count_per_sq_km,0.967675
1092,pharmacy_count_per_sq_km,supermarket_count_per_sq_km,0.945503
1650,theater_count_per_sq_km,venue_count_per_sq_km,0.920259
1458,social_club_count_per_sq_km,spaeti_count_per_sq_km,0.909688
724,kindergarten_count_per_sq_km,supermarket_count_per_sq_km,0.905527
474,gallery_count_per_sq_km,hotel_count_per_sq_km,0.897039
500,gallery_count_per_sq_km,venue_count_per_sq_km,0.892896
479,gallery_count_per_sq_km,museum_count_per_sq_km,0.886656
495,gallery_count_per_sq_km,theater_count_per_sq_km,0.875516


,feature_1,feature_2,correlation
1,kindergarten_count_per_1000,school_count_per_1000,0.568598


,feature_1,feature_2,correlation
39,avg_dist_5_bakery_m,avg_dist_5_pharmacy_m,0.919639
609,avg_dist_3_gas_station_m,avg_dist_3_gas_station_electric_m,0.913125
260,avg_dist_5_pharmacy_m,avg_dist_5_supermarket_m,0.903750
101,avg_dist_3_dental_office_m,avg_dist_5_pharmacy_m,0.900262
34,avg_dist_5_bakery_m,avg_dist_3_dental_office_m,0.887182
355,avg_dist_5_spaeti_m,avg_dist_5_transport_station_m,0.881571
291,avg_dist_3_playground_m,avg_dist_5_supermarket_m,0.878610
452,avg_dist_5_transport_station_m,avg_dist_3_food_market_m,0.876308
40,avg_dist_5_bakery_m,avg_dist_3_playground_m,0.874769
262,avg_dist_5_pharmacy_m,avg_dist_5_transport_station_m,0.870161


## Export final exploration dataset

The cleaned feature table is now exported as a CSV file for downstream tasks such as:

- exploratory analysis
- clustering
- feature selection
- machine learning modeling

This exported dataset contains only the final engineered features and excludes redundant raw variables.

In [27]:
final_output_path = Path("../data/neighborhood_features_exploration.csv")
df_model.to_csv(final_output_path, index=False)
print(f"Saved final exploration table to: {final_output_path}")

Saved final exploration table to: ../data/neighborhood_features_exploration.csv


## Feature engineering logic

Feature transformations follow four principles:

- **raw counts** for initial inspection
- **population density** for resident-facing amenities
- **spatial density** for point-based infrastructure and activity layers
- **area share** for land-use signals such as parks and playgrounds
- **proximity features** based on centroid-to-amenity distance, using different values of k depending on amenity frequency

This keeps the feature space interpretable while preparing it for later clustering and neighborhood-type discovery.